# KOL and Werewolf Targeting Analysis

This notebook analyzes `filtered_labeled_games_50.json` with an emphasis on the relationship between **key opinion leaders (KOLs)** and **Werewolf targeting**. It is designed for exploratory analysis and conference-paper figure production.

Main goals:
- characterize the dataset at game, player, and window level
- quantify how strongly KOL status and discussion leadership correlate with Werewolf targeting
- produce polished visualizations suitable for papers
- provide a few supplementary analyses beyond the main KOL-target question

## Setup

This notebook expects common analysis packages: `pandas`, `numpy`, `matplotlib`, `seaborn`, and `scipy`. If one is missing, install it in your environment before running the notebook.

In [ ]:
from pathlib import Path
import json
import math

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.labelweight": "bold",
    "legend.frameon": False,
})

DATA_PATH = Path("filtered_labeled_games_50.json")
FIG_DIR = Path("kol_target_paper_figures")
FIG_DIR.mkdir(exist_ok=True)

with DATA_PATH.open("r", encoding="utf-8") as f:
    games = json.load(f)

len(games)

## Flatten the JSON into analysis tables

In [ ]:
player_rows = []
window_rows = []
game_rows = []

for game in games:
    game_id = game["Game_ID"]
    windows = game.get("windows", [])
    profiles = game.get("player_profiles", [])

    kol_profile = next((p for p in profiles if p.get("influence") == "kol"), None)
    kol_player = kol_profile.get("player") if kol_profile else None

    for w in windows:
        window_rows.append({
            "game_id": game_id,
            "window_id": w.get("window_id"),
            "start_idx": w.get("start_merged_turn_index"),
            "end_idx": w.get("end_merged_turn_index"),
            "targeted_player": w.get("targeted_player"),
            "discussion_leader": w.get("discussion_leader"),
            "target_is_leader": w.get("targeted_player") == w.get("discussion_leader"),
            "target_is_group": w.get("targeted_player") == "Group",
            "target_is_none": w.get("targeted_player") == "None",
            "kol_player": kol_player,
            "target_is_kol": kol_player is not None and w.get("targeted_player") == kol_player,
            "leader_is_kol": kol_player is not None and w.get("discussion_leader") == kol_player,
            "window_len": (w.get("end_merged_turn_index", 0) - w.get("start_merged_turn_index", 0) + 1),
        })

    for p in profiles:
        player_rows.append({
            "game_id": game_id,
            "player": p.get("player"),
            "end_role": p.get("endRole"),
            "influence": p.get("influence"),
            "is_kol": p.get("influence") == "kol",
            "is_low_influence": p.get("influence") == "low_influence",
            "discussion_leader_count": p.get("discussion_leader_count", 0),
            "most_used_strategy": p.get("most_used_strategy"),
            "werewolf_target_count": p.get("werewolf_target_count", 0),
            "werewolf_target_rank": p.get("werewolf_target_rank"),
            "voted_werewolf": p.get("voted_werewolf"),
            "openness": p.get("personalities", {}).get("openness"),
            "conscientiousness": p.get("personalities", {}).get("conscientiousness"),
            "extraversion": p.get("personalities", {}).get("extraversion"),
            "agreeableness": p.get("personalities", {}).get("agreeableness"),
            "neuroticism": p.get("personalities", {}).get("neuroticism"),
        })

    game_rows.append({
        "game_id": game_id,
        "n_players": len(game.get("playerNames", [])),
        "n_windows": len(windows),
        "kol_player": kol_player,
        "kol_target_count": kol_profile.get("werewolf_target_count", 0) if kol_profile else np.nan,
        "kol_target_rank": kol_profile.get("werewolf_target_rank") if kol_profile else np.nan,
        "kol_leader_count": kol_profile.get("discussion_leader_count", 0) if kol_profile else np.nan,
        "windows_targeting_kol": sum(1 for w in windows if kol_player and w.get("targeted_player") == kol_player),
        "windows_where_leader_is_target": sum(1 for w in windows if w.get("targeted_player") == w.get("discussion_leader")),
    })

player_df = pd.DataFrame(player_rows)
window_df = pd.DataFrame(window_rows)
game_df = pd.DataFrame(game_rows)

player_df.head()

## Dataset overview

In [ ]:
overview = {
    "games": len(game_df),
    "players": len(player_df),
    "windows": len(window_df),
    "mean_windows_per_game": round(game_df["n_windows"].mean(), 2),
    "median_windows_per_game": float(game_df["n_windows"].median()),
    "mean_players_per_game": round(game_df["n_players"].mean(), 2),
}
overview

In [ ]:
display(player_df["influence"].value_counts().rename_axis("influence").to_frame("count"))
display(player_df["end_role"].value_counts().rename_axis("end_role").to_frame("count"))

## Main question: are KOLs targeted more often by Werewolves?

We analyze this at two levels:
1. **player-level**: do players with more discussion leadership end up being targeted more?
2. **game-level**: how often is the KOL the top Werewolf target in a game?

In [ ]:
pearson_r, pearson_p = stats.pearsonr(player_df["discussion_leader_count"], player_df["werewolf_target_count"])
spearman_rho, spearman_p = stats.spearmanr(player_df["discussion_leader_count"], player_df["werewolf_target_count"])

summary_stats = pd.Series({
    "pearson_r": pearson_r,
    "pearson_p": pearson_p,
    "spearman_rho": spearman_rho,
    "spearman_p": spearman_p,
    "games_where_kol_rank_1": int((game_df["kol_target_rank"] == 1).sum()),
    "games_total": int(len(game_df)),
    "share_games_kol_rank_1": float((game_df["kol_target_rank"] == 1).mean()),
    "avg_windows_targeting_kol": float(game_df["windows_targeting_kol"].mean()),
    "avg_leader_target_overlap": float(game_df["windows_where_leader_is_target"].sum() / len(window_df)),
})
summary_stats

## Figure 1. Discussion leadership vs Werewolf target count

This is the core player-level figure. It shows the relationship between how often a player leads discussion and how often the Werewolf targets them.

In [ ]:
palette = {"kol": "#b22222", "normal": "#4c78a8", "low_influence": "#59a14f"}
fig, ax = plt.subplots(figsize=(7.4, 5.6))
sns.scatterplot(
    data=player_df,
    x="discussion_leader_count",
    y="werewolf_target_count",
    hue="influence",
    palette=palette,
    s=85,
    alpha=0.85,
    ax=ax,
)
sns.regplot(
    data=player_df,
    x="discussion_leader_count",
    y="werewolf_target_count",
    scatter=False,
    color="black",
    line_kws={"linewidth": 2.2, "alpha": 0.9},
    ax=ax,
)
ax.set_title("Discussion leadership strongly tracks Werewolf targeting")
ax.set_xlabel("Discussion leader count")
ax.set_ylabel("Werewolf target count")
ax.text(0.98, 0.04, f"Pearson r = {pearson_r:.3f}\nSpearman ρ = {spearman_rho:.3f}",
        transform=ax.transAxes, ha="right", va="bottom", fontsize=11,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="0.8"))
fig.tight_layout()
fig.savefig(FIG_DIR / "figure1_leader_vs_target_scatter.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "figure1_leader_vs_target_scatter.png", bbox_inches="tight")
plt.show()

## Figure 2. Werewolf target count by influence class

This summarizes the distributional difference between KOLs, normal players, and low-influence players.

In [ ]:
fig, ax = plt.subplots(figsize=(7.2, 5.6))
order = ["kol", "normal", "low_influence"]
sns.violinplot(
    data=player_df,
    x="influence",
    y="werewolf_target_count",
    order=order,
    palette=palette,
    inner=None,
    cut=0,
    linewidth=1.1,
    ax=ax,
)
sns.boxplot(
    data=player_df,
    x="influence",
    y="werewolf_target_count",
    order=order,
    width=0.24,
    showcaps=True,
    boxprops={"facecolor": "white", "zorder": 3},
    showfliers=False,
    whiskerprops={"linewidth": 1.3},
    medianprops={"color": "black", "linewidth": 2},
    ax=ax,
)
ax.set_title("KOLs are targeted more often than other players")
ax.set_xlabel("Influence label")
ax.set_ylabel("Werewolf target count")
fig.tight_layout()
fig.savefig(FIG_DIR / "figure2_target_count_by_influence.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "figure2_target_count_by_influence.png", bbox_inches="tight")
plt.show()

## Figure 3. KOL target rank across games

This game-level figure shows how often the KOL is the first-ranked Werewolf target.

In [ ]:
rank_counts = game_df["kol_target_rank"].value_counts().sort_index()
rank_share = rank_counts / rank_counts.sum()

fig, ax = plt.subplots(figsize=(7.0, 5.2))
bar = sns.barplot(x=rank_counts.index.astype(int), y=rank_counts.values, color="#7a5195", ax=ax)
for i, (count, share) in enumerate(zip(rank_counts.values, rank_share.values)):
    ax.text(i, count + 0.35, f"{count}\n({share:.0%})", ha="center", va="bottom", fontsize=11)
ax.set_title("The KOL is usually the top Werewolf target")
ax.set_xlabel("KOL werewolf-target rank (1 = highest target count)")
ax.set_ylabel("Number of games")
fig.tight_layout()
fig.savefig(FIG_DIR / "figure3_kol_rank_distribution.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "figure3_kol_rank_distribution.png", bbox_inches="tight")
plt.show()

## Figure 4. How often is the discussion leader also the target?

This figure shows the game-level rate at which Werewolf targeting aligns with the current discussion leader.

In [ ]:
game_df = game_df.assign(leader_target_overlap_rate = game_df["windows_where_leader_is_target"] / game_df["n_windows"])
fig, ax = plt.subplots(figsize=(7.0, 5.2))
sns.histplot(game_df["leader_target_overlap_rate"], bins=10, color="#f28e2b", edgecolor="white", ax=ax)
ax.axvline(game_df["leader_target_overlap_rate"].mean(), color="black", linestyle="--", linewidth=2, label="mean")
ax.set_title("Leader-target overlap is common across games")
ax.set_xlabel("Rate of windows where targeted player = discussion leader")
ax.set_ylabel("Number of games")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "figure4_leader_target_overlap_rate.pdf", bbox_inches="tight")
fig.savefig(FIG_DIR / "figure4_leader_target_overlap_rate.png", bbox_inches="tight")
plt.show()

## Statistical tests

These tests make the main claims more reportable in a paper.

In [ ]:
kol_targets = player_df.loc[player_df["influence"] == "kol", "werewolf_target_count"]
non_kol_targets = player_df.loc[player_df["influence"] != "kol", "werewolf_target_count"]
normal_targets = player_df.loc[player_df["influence"] == "normal", "werewolf_target_count"]
low_targets = player_df.loc[player_df["influence"] == "low_influence", "werewolf_target_count"]

mannwhitney_kol_vs_nonkol = stats.mannwhitneyu(kol_targets, non_kol_targets, alternative="greater")
kruskal_all = stats.kruskal(kol_targets, normal_targets, low_targets)

contingency = pd.crosstab(player_df["influence"], player_df["werewolf_target_rank"] == 1)
chi2, chi2_p, dof, expected = stats.chi2_contingency(contingency)

pd.Series({
    "mannwhitney_kol_vs_nonkol_stat": mannwhitney_kol_vs_nonkol.statistic,
    "mannwhitney_kol_vs_nonkol_p": mannwhitney_kol_vs_nonkol.pvalue,
    "kruskal_stat": kruskal_all.statistic,
    "kruskal_p": kruskal_all.pvalue,
    "chi2_target_rank1_by_influence": chi2,
    "chi2_p": chi2_p,
    "games_kol_rank1_share": (game_df["kol_target_rank"] == 1).mean(),
})

## Supplementary analysis: personality traits and targeting

This is a lighter exploratory check beyond the main KOL question.

In [ ]:
trait_order = {"low": 0, "moderate": 1, "high": 2}
trait_cols = ["openness", "conscientiousness", "extraversion", "agreeableness", "neuroticism"]
trait_df = player_df.copy()
for col in trait_cols:
    trait_df[col + "_num"] = trait_df[col].map(trait_order)

trait_corrs = {}
for col in trait_cols:
    rho, p = stats.spearmanr(trait_df[col + "_num"], trait_df["werewolf_target_count"], nan_policy="omit")
    trait_corrs[col] = {"spearman_rho": rho, "p": p}

pd.DataFrame(trait_corrs).T.sort_values("spearman_rho", ascending=False)

## Figure export summary

In [ ]:
sorted(str(p) for p in FIG_DIR.iterdir())

## Suggested reporting language

You can adapt the following pattern for a paper:

- *Player-level discussion leadership was strongly positively associated with Werewolf targeting frequency (Pearson r = ..., Spearman ρ = ...).*
- *At the game level, the KOL was the top-ranked Werewolf target in ...% of games.*
- *KOL players also had substantially higher target counts than normal and low-influence players, suggesting that Werewolves preferentially orient persuasion toward influential table actors.*